# FINAL RECOVERY — one Colab, then decide TEST3

```text
B1 Linux E2T → B1 cache replay → B2 Agent.solve Tables → B2b K (optional)
→ B3 No-files → B4 static → BAKE → RECOVERY_READY
```

**STOP** at first failed gate. No Qwen3 / new prompts / rules / auto-submit.

If `RECOVERY_READY`: submit `STAGE3_E2T_RECOVERY.zip` manually as TEST3.


In [3]:
!git clone https://github.com/BeediGoua/TEST-PROJET-ML.git
from IPython.display import display, HTML

file_path = "/content/TEST-PROJET-ML/audit_independent/colab_final_recovery.ipynb"
print(f"Fichier : {file_path}")
print("Fichiers (gauche) → TEST-PROJET-ML → audit_independent → colab_final_recovery.ipynb")



fatal: destination path 'TEST-PROJET-ML' already exists and is not an empty directory.
Fichier : /content/TEST-PROJET-ML/audit_independent/colab_final_recovery.ipynb
Fichiers (gauche) → TEST-PROJET-ML → audit_independent → colab_final_recovery.ipynb


## 00 — Setup

In [4]:

from pathlib import Path
import sys, os, platform, json, tempfile, time
assert platform.system()=="Linux", "FINAL RECOVERY requires Colab Linux"
# !git clone https://github.com/BeediGoua/TEST-PROJET-ML.git
PROJECT_ROOT = Path("/content/TEST-PROJET-ML")
assert (PROJECT_ROOT/"audit_independent"/"audit.py").exists()
AUDIT = PROJECT_ROOT/"audit_independent"
sys.path[:0] = [str(AUDIT), str(PROJECT_ROOT), str(AUDIT/"diagnostic")]
os.chdir(AUDIT)
!pip -q install torch transformers accelerate pandas numpy scipy scikit-learn sympy pyarrow 2>/dev/null || true
from gpu_experiments import official
from build_experimental import build_test1_e2t, build_test1_alias
from pretest3_common import EXP, load_path, AUDIT_OUT, DIAG_OUT, table_tasks, load_cache, is_real_qwen_cache
import audit as audit_mod
from src.evaluation.core import _materialize_all
import final_recovery_harness as H
from bake_stage3_e2t_recovery import bake
build_test1_alias(); build_test1_e2t(); official()
e2t = load_path(EXP/"test1_e2t_dsh.py", "e2t")
stage3 = audit_mod.load("stage3")
CACHE = AUDIT_OUT/"generations_34d6ece55b3c092a.json"
assert CACHE.exists(), "Upload frozen cache into audit_independent/results/"
cache = load_cache(CACHE); assert is_real_qwen_cache(cache)
print("OK setup", platform.platform())


OK setup Linux-6.6.122+-x86_64-with-glibc2.39


## B1 — Linux E2T contract (STOP if fail)

In [5]:

tasks = table_tasks(); t0 = tasks[0]
with tempfile.TemporaryDirectory() as td:
    fm = _materialize_all([t0], td)
    H.b1_linux_contract(e2t, stage3, t0, fm)


PASS B1_linux_contract {'loop': '[6.0, None, 42.0, 9.0]'}


## B1 — Cache replay 400 programs (task-level gate)

In [6]:

OUTER = float(os.environ.get("DSH_OUTER_MANY", "7.5"))
gold = {t["id"]: t for t in table_tasks()}
with tempfile.TemporaryDirectory() as td:
    fm = _materialize_all(list(gold.values()), td)
    m = H.b1_cache_replay(e2t, stage3, cache, gold, fm, outer=OUTER)
print(m)


PASS B1_cache_replay {'program_agree': 1.0, 'task_vote_agree': 1.0, 'stage3_accuracy': 0.7875, 'e2t_accuracy': 0.7875, 'none_where_e0_finite': 0, 'bad_numeric': 0, 'p50': 1.042698822000034, 'p95': 1.2723281340000767, 'max': 1.3919715229999383, 'outer': 7.5}
{'program_agree': 1.0, 'task_vote_agree': 1.0, 'stage3_accuracy': 0.7875, 'e2t_accuracy': 0.7875, 'none_where_e0_finite': 0, 'bad_numeric': 0, 'p50': 1.042698822000034, 'p95': 1.2723281340000767, 'max': 1.3919715229999383, 'outer': 7.5}


## B2 — REAL Agent.solve Tables (forced fallback) — GATE PRINCIPAL

In [7]:

import torch
assert torch.cuda.is_available(), "B2 needs CUDA"
# Bake recovery agent (Stage3+E2T); Stage3 historical stays local-only for reference
agent_path = bake(outer_many=OUTER)
print("baked", agent_path)
# Load agents
A = audit_mod.load("stage3")  # historical exec OK locally
Bmod = H.load_mod(agent_path, "stage3_e2t_rec")
# Force same model
A.MODEL = Bmod.MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
os.environ["DSH_FORCE_FALLBACK"] = "1"
os.environ["DSH_OUTER_MANY"] = str(OUTER)
agent_a = A.Agent()
agent_b = Bmod.Agent()
tables = table_tasks()
TD_TABLES = tempfile.mkdtemp(prefix="final_tables_")
fm = _materialize_all(tables, TD_TABLES)
batch_tasks = []
for t in tables:
    tt = dict(t)
    tt["files"] = fm[t["id"]]
    batch_tasks.append(tt)
df_t, sum_t = H.run_solve_compare("Stage3", agent_a, "E2T", agent_b, batch_tasks, batch_size=8, force_fallback=True)
df_t.to_csv(DIAG_OUT/"final_b2_tables.csv", index=False)
print(sum_t)


baked /content/TEST-PROJET-ML/audit_independent/STAGE3_E2T_RECOVERY/agent.py


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[init] modèle Qwen/Qwen2.5-Coder-1.5B-Instruct sur cuda:0, échantillons 4, budget 29.0s


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[init] modèle Qwen/Qwen2.5-Coder-1.5B-Instruct sur cuda:0, échantillons 4, budget 29.0s
[solve] prep 0.3s | glouton 0.0s | réparation 0.0s | échantillons 0.0s | total 0.3s
[solve] prep 0.3s | glouton 0.0s | réparation 0.0s | échantillons 0.0s | total 0.3s
[solve] prep 0.3s | glouton 0.0s | réparation 0.0s | échantillons 0.0s | total 0.3s
[solve] prep 0.3s | glouton 0.0s | réparation 0.0s | échantillons 0.0s | total 0.3s
[solve] prep 0.2s | glouton 0.0s | réparation 0.0s | échantillons 0.0s | total 0.2s
[solve] prep 0.2s | glouton 0.0s | réparation 0.0s | échantillons 0.0s | total 0.2s
[solve] prep 0.3s | glouton 0.0s | réparation 0.0s | échantillons 0.0s | total 0.3s
[solve] prep 0.4s | glouton 0.0s | réparation 0.0s | échantillons 0.0s | total 0.4s
[solve] prep 0.3s | glouton 0.0s | réparation 0.0s | échantillons 0.0s | total 0.3s
[solve] prep 0.3s | glouton 0.0s | réparation 0.0s | échantillons 0.0s | total 0.3s
[solve] prep 0.3s | glouton 0.0s | réparation 0.0s | échantillons 0.0s |

## B2b — workload K (optional, cheap stress)

In [8]:

# Quantify overhead only; does not veto unless a batch >40s (harness already STOPS)
Ks = [1, 2, 4, 8]
stress = []
# use first 8 tables; mark K as fallback by FORCE_FALLBACK already on all — simulate K by solving subsets
for K in Ks:
    sub = batch_tasks[:K]
    # pad with deterministic answered dummies? skip — just time K-task solve
    t0 = time.perf_counter(); agent_b.solve(sub); dt = time.perf_counter()-t0
    stress.append(dict(K=K, seconds=dt))
    print("K", K, "s", dt)
    if dt > 40:
        H.stop(f"B2b K={K} >40s")
import pandas as pd
pd.DataFrame(stress).to_csv(DIAG_OUT/"final_b2b_stress.csv", index=False)
H.mark("B2b_stress", **{f"K{s['K']}": s["seconds"] for s in stress})


[solve] prep 0.0s | glouton 0.0s | réparation 0.0s | échantillons 0.0s | total 0.0s
K 1 s 0.026664458999903218
[solve] prep 0.1s | glouton 0.0s | réparation 0.0s | échantillons 0.0s | total 0.1s
K 2 s 0.07382015500002126
[solve] prep 0.2s | glouton 0.0s | réparation 0.0s | échantillons 0.0s | total 0.2s
K 4 s 0.16016970199996194
[solve] prep 0.3s | glouton 0.0s | réparation 0.0s | échantillons 0.0s | total 0.3s
K 8 s 0.2950237030000835
PASS B2b_stress {'K1': 0.026664458999903218, 'K2': 0.07382015500002126, 'K4': 0.16016970199996194, 'K8': 0.2950237030000835}


## B3 — No-files uncovered by calc_rules

In [12]:
import math, time, os
import pandas as pd

# Rebuild nofiles if needed
HANDOFF = PROJECT_ROOT / "CODEX_AUDIT_HANDOFF"
all_tasks = json.loads((HANDOFF / "dev.json").read_text(encoding="utf-8"))
nofiles = []
for t in all_tasks:
    if t.get("files"):
        continue
    try:
        rv = A.calc_rules(t["objective"])
    except Exception:
        rv = None
    if rv is None or not (isinstance(rv, (int, float)) and math.isfinite(float(rv))):
        nofiles.append(t)

TD_NF = tempfile.mkdtemp(prefix="final_nofiles_")
fm = _materialize_all(nofiles, TD_NF)
nf_tasks = [{**t, "files": fm[t["id"]]} for t in nofiles]

os.environ.pop("DSH_FORCE_FALLBACK", None)
rows = []
for start in range(0, len(nf_tasks), 8):
    batch = nf_tasks[start:start+8]
    t0 = time.perf_counter(); ra = agent_a.solve(batch); ta = time.perf_counter() - t0
    t0 = time.perf_counter(); rb = agent_b.solve(batch); tb = time.perf_counter() - t0
    by_a = {r["id"]: r["answer"] for r in ra}
    by_b = {r["id"]: r["answer"] for r in rb}
    for t in batch:
        ans, tol = float(t["answer"]), float(t["tol"])
        va, vb = by_a.get(t["id"]), by_b.get(t["id"])
        rows.append(dict(
            id=t["id"], gold=ans, a=va, b=vb,
            a_ok=va is not None and abs(float(va)-ans) <= tol,
            b_ok=vb is not None and abs(float(vb)-ans) <= tol,
            time_a=ta, time_b=tb,
        ))

df = pd.DataFrame(rows)
path = DIAG_OUT / "final_b3_nofiles.csv"
df.to_csv(path, index=False)
print("saved", path)
print("Stage3", df.a_ok.mean(), "E2T", df.b_ok.mean())
lost = df[(df.a_ok == True) & (df.b_ok == False)]
print("Stage3 OK, E2T KO:", len(lost))
print(lost[["id", "gold", "a", "b"]].to_string(index=False))

[solve] prep 0.0s | glouton 5.4s | réparation 7.4s | échantillons 9.9s | total 22.7s
[solve] prep 0.0s | glouton 16.2s | réparation 6.7s | échantillons 5.4s | total 28.4s
[solve] prep 0.0s | glouton 5.1s | réparation 5.6s | échantillons 8.8s | total 19.4s
[solve] prep 0.0s | glouton 16.0s | réparation 9.8s | échantillons 0.0s | total 25.8s
[solve] prep 0.0s | glouton 3.0s | réparation 0.0s | échantillons 6.2s | total 9.3s
[solve] prep 0.0s | glouton 14.0s | réparation 0.0s | échantillons 14.1s | total 28.1s
[solve] prep 0.0s | glouton 4.4s | réparation 4.8s | échantillons 8.4s | total 17.6s
[solve] prep 0.0s | glouton 15.4s | réparation 7.6s | échantillons 0.0s | total 23.0s
[solve] prep 0.0s | glouton 4.6s | réparation 0.0s | échantillons 8.2s | total 12.7s
[solve] prep 0.0s | glouton 15.5s | réparation 0.0s | échantillons 12.1s | total 27.6s
saved /content/TEST-PROJET-ML/audit_independent/diagnostic/results/final_b3_nofiles.csv
Stage3 0.975 E2T 0.9
Stage3 OK, E2T KO: 4
   id  gold   

## B4 — Static / resource check + BAKE decision

In [13]:

rep = H.b4_static_and_resources(agent_path, AUDIT/"STAGE3_E2T_RECOVERY"/"dsh.py")
# Re-bake fresh zip after all greens
bake(outer_many=OUTER)
H.finalize_ready()
print("Download STAGE3_E2T_RECOVERY.zip and submit TEST3 manually if you accept RECOVERY_READY.")


PASS B4_static {'agent_file': 'agent.py', 'exec_upload_violation': False, 'sympy_prompt_runtime_mismatch': True, 'onehot_prompt_runtime_mismatch': True, 'no_imports_example_mismatch': True, 'prompt_runtime_mismatch_count': 3, 'zip_bytes': 44043, 'nfiles': 2}
# FINAL RECOVERY

**RECOVERY_READY**

Gates passed:
- B1_linux_contract: {'loop': '[6.0, None, 42.0, 9.0]'}
- B1_cache_replay: {'program_agree': 1.0, 'task_vote_agree': 1.0, 'stage3_accuracy': 0.7875, 'e2t_accuracy': 0.7875, 'none_where_e0_finite': 0, 'bad_numeric': 0, 'p50': 1.042698822000034, 'p95': 1.2723281340000767, 'max': 1.3919715229999383, 'outer': 7.5}
- solve_E2T_vs_Stage3: {'n': 80, 'acc_a': 1.0, 'acc_b': 1.0, 'task_agree': 1.0, 'max_batch_a': 0.36216077099993527, 'max_batch_b': 0.3592274989999851}
- B2b_stress: {'K1': 0.026664458999903218, 'K2': 0.07382015500002126, 'K4': 0.16016970199996194, 'K8': 0.2950237030000835}
- B4_static: {'agent_file': 'agent.py', 'exec_upload_violation': False, 'sympy_prompt_runtime_mismatch'